# 第5章 深度学习计算

## 5.1 层和块

In [1]:
import torch
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
X = torch.rand(2, 20)
print(net(X))

tensor([[-0.0712, -0.1709,  0.1206, -0.0112, -0.1543,  0.0958,  0.2252, -0.2007,
          0.3352, -0.3098],
        [-0.0453, -0.1059,  0.1614,  0.0109, -0.1247, -0.0033,  0.1854, -0.2265,
          0.4736, -0.3625]], grad_fn=<AddmmBackward0>)


### 5.1.1 自定义块

In [2]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.out = nn.Linear(256, 10)

    def forward(self, X):
        return self.out(F.relu(self.hidden(X)))

net = MLP()
print(net(X))

tensor([[ 0.0012, -0.2398, -0.1384, -0.0264,  0.1521, -0.0116,  0.0179,  0.0109,
         -0.2309,  0.0577],
        [ 0.0219, -0.2112,  0.0173, -0.0983,  0.1921, -0.0401,  0.0175,  0.0378,
         -0.2012,  0.1295]], grad_fn=<AddmmBackward0>)


### 5.1.2 顺序块

In [3]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            self._modules[str(idx)] = module

    def forward(self, X):
        for block in self._modules.values():
            X = block(X)
        return X

net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
print(net(X))

tensor([[ 0.0460,  0.1237,  0.0332,  0.0316, -0.0725,  0.2356, -0.2276,  0.0005,
          0.0673,  0.2603],
        [ 0.1108,  0.2079, -0.0066,  0.0850, -0.0870,  0.2495, -0.3846, -0.0301,
          0.0621,  0.3475]], grad_fn=<AddmmBackward0>)


### 5.1.3 在前向传播函数中执行代码

In [4]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        X = self.linear(X)
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

net = FixedHiddenMLP()
print(net(X))

tensor(-0.1526, grad_fn=<SumBackward0>)


In [5]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(),
                                 nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)

    def forward(self, X):
        return self.linear(self.net(X))

chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
print(chimera(X))

tensor(-0.1993, grad_fn=<SumBackward0>)


## 5.2 参数管理

In [6]:
import torch
from torch import nn

net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
X = torch.rand(size=(2, 4))
print(net(X))

tensor([[-0.0464],
        [-0.0482]], grad_fn=<AddmmBackward0>)


### 5.2.1 参数访问

In [7]:
print(net[2].state_dict())
print(type(net[2].bias))
print(net[2].bias)
print(net[2].bias.data)
print(net[2].weight.grad == None)

OrderedDict({'weight': tensor([[-0.3083, -0.3213, -0.0399, -0.0786, -0.2899,  0.2725,  0.1807,  0.0908]]), 'bias': tensor([0.0225])})
<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([0.0225], requires_grad=True)
tensor([0.0225])
True


In [8]:
print(*[(name, param.shape) for name, param in net[0].named_parameters()])
print(*[(name, param.shape) for name, param in net.named_parameters()])
print(net.state_dict()['2.bias'].data)

('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))
tensor([0.0225])


In [9]:
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                         nn.Linear(8, 4), nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        net.add_module(f'block {i}', block1())
    return net

rgnet = nn.Sequential(block2(), nn.Linear(4, 1))
print(rgnet(X))
print(rgnet)
print(rgnet[0][1][0].bias.data)

tensor([[0.2667],
        [0.2668]], grad_fn=<AddmmBackward0>)
Sequential(
  (0): Sequential(
    (block 0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)
tensor([-0.2906,  0.2346, -0.2620,  0.0483,  0.4342, -0.1318,  0.2586, -0.1645])


### 5.2.2 参数初始化

In [10]:
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, mean=0, std=0.01)
        nn.init.zeros_(m.bias)

net.apply(init_normal)
print(net[0].weight.data[0], net[0].bias.data[0])

def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 1)
        nn.init.zeros_(m.bias)

net.apply(init_constant)
print(net[0].weight.data[0], net[0].bias.data[0])

tensor([-0.0103,  0.0039,  0.0019, -0.0023]) tensor(0.)
tensor([1., 1., 1., 1.]) tensor(0.)


In [11]:
def init_xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)

def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 42)

net[0].apply(init_xavier)
net[2].apply(init_42)
print(net[0].weight.data[0])
print(net[2].weight.data)

tensor([ 0.5849,  0.2487, -0.5283, -0.4752])
tensor([[42., 42., 42., 42., 42., 42., 42., 42.]])


In [12]:
def my_init(m):
    if type(m) == nn.Linear:
        print("Init", *[(name, param.shape)
                        for name, param in m.named_parameters()][0])
        nn.init.uniform_(m.weight, -10, 10)
        m.weight.data *= m.weight.data.abs() >= 5

net.apply(my_init)
print(net[0].weight[:2])

net[0].weight.data[:] += 1
net[0].weight.data[0, 0] = 42
print(net[0].weight.data[0])

Init weight torch.Size([8, 4])
Init weight torch.Size([1, 8])
tensor([[ 0.0000,  5.5873,  9.0620, -0.0000],
        [ 0.0000, -0.0000, -7.2180, -6.6117]], grad_fn=<SliceBackward0>)
tensor([42.0000,  6.5873, 10.0620,  1.0000])


### 5.2.3 参数绑定

In [13]:
shared = nn.Linear(8, 8)
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                    shared, nn.ReLU(),
                    shared, nn.ReLU(),
                    nn.Linear(8, 1))
net(X)
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0, 0] = 100
print(net[2].weight.data[0] == net[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])


## 5.4 自定义层

### 5.4.1 不带参数的层

In [14]:
import torch
import torch.nn.functional as F
from torch import nn

class CenteredLayer(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, X):
        return X - X.mean()

layer = CenteredLayer()
print(layer(torch.FloatTensor([1, 2, 3, 4, 5])))

net = nn.Sequential(nn.Linear(8, 128), CenteredLayer())
Y = net(torch.rand(4, 8))
print(Y.mean())

tensor([-2., -1.,  0.,  1.,  2.])
tensor(4.1910e-09, grad_fn=<MeanBackward0>)


### 5.4.2 带参数的层

In [15]:
class MyLinear(nn.Module):
    def __init__(self, in_units, units):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(in_units, units))
        self.bias = nn.Parameter(torch.randn(units,))

    def forward(self, X):
        linear = torch.matmul(X, self.weight.data) + self.bias.data
        return F.relu(linear)

linear = MyLinear(5, 3)
print(linear.weight)
print(linear(torch.rand(2, 5)))

net = nn.Sequential(MyLinear(64, 8), MyLinear(8, 1))
print(net(torch.rand(2, 64)))

Parameter containing:
tensor([[-0.9857,  0.3979,  0.0155],
        [ 1.8633, -1.8883,  0.7502],
        [ 0.6067,  0.6975, -0.0242],
        [ 0.9674, -2.1336,  1.9365],
        [ 0.7718, -1.0620, -0.7265]], requires_grad=True)
tensor([[3.4296, 0.0000, 0.0000],
        [2.5986, 1.4599, 0.0000]])
tensor([[0.0000],
        [0.0651]])


## 5.5 读写文件

### 5.5.1 加载和保存张量

In [16]:
import torch
from torch import nn
from torch.nn import functional as F

x = torch.arange(4)
torch.save(x, 'x-file')
x2 = torch.load('x-file')
print(x2)

y = torch.zeros(4)
torch.save([x, y], 'x-files')
x2, y2 = torch.load('x-files')
print((x2, y2))

mydict = {'x': x, 'y': y}
torch.save(mydict, 'mydict')
mydict2 = torch.load('mydict')
print(mydict2)

tensor([0, 1, 2, 3])
(tensor([0, 1, 2, 3]), tensor([0., 0., 0., 0.]))
{'x': tensor([0, 1, 2, 3]), 'y': tensor([0., 0., 0., 0.])}


### 5.5.2 加载和保存模型参数

In [17]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.output = nn.Linear(256, 10)

    def forward(self, x):
        return self.output(F.relu(self.hidden(x)))

net = MLP()
X = torch.randn(size=(2, 20))
Y = net(X)

torch.save(net.state_dict(), 'mlp.params')

clone = MLP()
clone.load_state_dict(torch.load('mlp.params'))
print(clone.eval())

Y_clone = clone(X)
print(Y_clone == Y)

MLP(
  (hidden): Linear(in_features=20, out_features=256, bias=True)
  (output): Linear(in_features=256, out_features=10, bias=True)
)
tensor([[True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True]])


## 5.6 GPU

In [18]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


### 5.6.1 计算设备

In [19]:
import torch
from torch import nn

print(torch.device('cpu'), torch.device('cuda'), torch.device('cuda:1'))
print(torch.cuda.device_count())

def try_gpu(i=0):
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')

def try_all_gpus():
    devices = [torch.device(f'cuda:{i}')
               for i in range(torch.cuda.device_count())]
    return devices if devices else [torch.device('cpu')]

print(try_gpu(), try_gpu(10), try_all_gpus())

cpu cuda cuda:1
0
cpu cpu [device(type='cpu')]


### 5.6.2 张量与GPU

In [20]:
x = torch.tensor([1, 2, 3])
print(x.device)

X = torch.ones(2, 3, device=try_gpu())
print(X)

Y = torch.rand(2, 3, device=try_gpu(1))
print(Y)

cpu
tensor([[1., 1., 1.],
        [1., 1., 1.]])
tensor([[0.9207, 0.0520, 0.8550],
        [0.3904, 0.2858, 0.9114]])


In [21]:
if torch.cuda.device_count() >= 2:
    Z = X.cuda(1)
    print(X)
    print(Z)
    print(Y + Z)
    print(Z.cuda(1) is Z)
else:
    print(f"当前可用 GPU 数量: {torch.cuda.device_count()}，跳过双卡示例")

当前可用 GPU 数量: 0，跳过双卡示例


### 5.6.3 神经网络与GPU

In [22]:
net = nn.Sequential(nn.Linear(3, 1))
net = net.to(device=try_gpu())

print(net(X))
print(net[0].weight.data.device)

tensor([[0.1287],
        [0.1287]], grad_fn=<AddmmBackward0>)
cpu
